# Qwen-500 Fine-Tuning Skeleton

Barebones notebook for fine-tuning a Qwen 0.5B model on local CSV splits.

Installs required Python packages for training and evaluation (run only if your environment is missing dependencies).

In [ ]:
# If needed, uncomment:
# !pip install -q transformers datasets accelerate evaluate scikit-learn

Imports all libraries used in the notebook: data loading, tokenization, model training, and metrics.

In [ ]:
from pathlib import Path
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import evaluate


Defines all core configuration values, including model name, dataset paths, column names, and output directory.

In [ ]:
# --- Config ---
MODEL_ID = "Qwen/Qwen2.5-0.5B"  # swap if you use a different Qwen-500 checkpoint
TRAIN_CSV = str(Path("../datasets/splits/train.csv").resolve())
VAL_CSV = str(Path("../datasets/splits/val.csv").resolve())
TEXT_COL = "prompt"
LABEL_COL = "class"
MAX_LENGTH = 512
OUTPUT_DIR = "./qwen-500-finetuned"
SEED = 42


Detects whether CUDA GPU is available (such as T4), then configures precision and batch sizes accordingly.

In [ ]:
# --- Runtime / Device Setup (uses T4 if available, but does not force) ---
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    USE_FP16 = True
    USE_BF16 = False
    TRAIN_BS = 8
    EVAL_BS = 16
    NUM_WORKERS = 2
    print(f"CUDA available: {device_name}")
else:
    device_name = "cpu"
    USE_FP16 = False
    USE_BF16 = False
    TRAIN_BS = 2
    EVAL_BS = 4
    NUM_WORKERS = 0
    print("CUDA not available. Using CPU.")

print(
    f"Device={device_name} | fp16={USE_FP16} | bf16={USE_BF16} | "
    f"train_bs={TRAIN_BS} | eval_bs={EVAL_BS}"
)


Loads the train and validation CSV files into a Hugging Face DatasetDict.

In [ ]:
dataset = load_dataset(
    "csv",
    data_files={"train": TRAIN_CSV, "validation": VAL_CSV},
)
dataset


Builds tokenizer preprocessing logic and tokenizes the datasets, while converting labels to integer class IDs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(batch):
    texts = [str(x) for x in batch[TEXT_COL]]
    labels = [int(float(x)) for x in batch[LABEL_COL]]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = labels
    return enc

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
tokenized


Initializes the classification model, metric functions, training arguments, and Hugging Face Trainer object.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=2,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    dataloader_num_workers=NUM_WORKERS,
    fp16=USE_FP16,
    bf16=USE_BF16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


Runs model fine-tuning, then evaluates the model on the validation split.

In [ ]:
# Train
trainer.train()

# Evaluate
metrics = trainer.evaluate()
metrics

Saves the trained model weights and tokenizer files to the configured output directory.

In [ ]:
# Save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved model to {OUTPUT_DIR}")